# Exploratory Data Analysis

Analysis of the SMVEC student dataset for the AI Peer Study Group Formation System.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
data_path = os.path.join("..", "..", "data_pipeline", "data", "students.json")
with open(data_path, "r") as f:
    students_raw = json.load(f)

df = pd.json_normalize(students_raw)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.info()
df.head(3)

## 1. Basic Statistics

In [ ]:
print("=== Descriptive Statistics ===")
print(df[["cgpa", "year"]].describe())

print("\n=== Department Distribution ===")
print(df["department"].value_counts())

print("\n=== Year Distribution ===")
print(df["year"].value_counts().sort_index())

print("\n=== Learning Pace Distribution ===")
print(df["learning_pace"].value_counts())

print("\n=== Archetype Distribution ===")
print(df["archetype"].value_counts())

## 2. CGPA Distribution by Department

In [ ]:
departments = df["department"].unique()
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, dept in enumerate(sorted(departments)):
    subset = df[df["department"] == dept]["cgpa"]
    axes[idx].hist(subset, bins=20, color=sns.color_palette()[idx], edgecolor="black", alpha=0.7)
    axes[idx].set_title(f"{dept} (n={len(subset)})")
    axes[idx].set_xlabel("CGPA")
    axes[idx].set_ylabel("Count")
    axes[idx].axvline(subset.mean(), color="red", linestyle="--", label=f"Mean={subset.mean():.2f}")
    axes[idx].legend()

fig.suptitle("CGPA Distribution by Department", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Skill Distribution Heatmap

In [ ]:
subjects = [
    "DSA", "OOP", "DBMS", "OS", "Networks",
    "Mathematics", "Physics", "English", "Machine Learning", "Web Development"
]

skill_matrix = np.zeros((len(df), len(subjects)))
for i, student in enumerate(students_raw):
    for j, subject in enumerate(subjects):
        for skill in student["skills"]:
            if skill["subject"] == subject:
                skill_matrix[i, j] = skill["self_rating"]
                break

skill_df = pd.DataFrame(skill_matrix, columns=subjects)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    skill_df.describe().loc[["mean", "std", "min", "25%", "50%", "75%", "max"]],
    annot=True, fmt=".2f", cmap="YlOrRd", ax=ax
)
ax.set_title("Skill Distribution Summary Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Archetype Distribution

In [ ]:
archetype_counts = df["archetype"].value_counts()
colors = sns.color_palette("Set2", len(archetype_counts))

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    archetype_counts.values,
    labels=archetype_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=140,
    pctdistance=0.85,
)
for autotext in autotexts:
    autotext.set_fontsize(10)
ax.set_title("Student Archetype Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Availability Patterns

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
slots = ["morning", "afternoon", "evening"]

avail_matrix = np.zeros((7, 3))
for student in students_raw:
    for entry in student["availability"]:
        day_idx = entry["day_of_week"]
        slot_idx = slots.index(entry["slot"])
        if entry["is_available"]:
            avail_matrix[day_idx, slot_idx] += 1

avail_pct = avail_matrix / len(students_raw) * 100
avail_df = pd.DataFrame(avail_pct, index=days, columns=[s.capitalize() for s in slots])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(avail_df, annot=True, fmt=".1f", cmap="Blues", ax=ax, vmin=0, vmax=100)
ax.set_title("Student Availability (% available per slot)", fontsize=14, fontweight="bold")
ax.set_ylabel("Day of Week")
ax.set_xlabel("Time Slot")
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
skill_means = skill_df.mean(axis=1)

avail_counts = []
for student in students_raw:
    count = sum(1 for a in student["availability"] if a["is_available"])
    avail_counts.append(count)

corr_df = pd.DataFrame({
    "CGPA": df["cgpa"].values,
    "Year": df["year"].values,
    "Skill Mean": skill_means.values,
    "Availability Count": avail_counts,
})

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()